# MODULES

In [1]:
!pip install pathlib
!pip install matplotlib

In [2]:
!nvidia-smi

Thu Sep 18 13:33:49 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.230.02             Driver Version: 535.230.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3060        Off | 00000000:01:00.0  On |                  N/A |
| 30%   47C    P2              39W / 170W |   4969MiB / 12288MiB |      4%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [3]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

1.13.1+cu117
11.7
NVIDIA GeForce RTX 3060


In [4]:
import cv2
import os
import math
import time
import random
import pathlib
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt

# GLOBAL VARIABLES

In [5]:
ROOT_DIR_IMAGES = '../Kaggle'

# IMAGES

In [6]:
def get_image_paths(root_dir, include='*', exclude=[]):
  paths = []
  
  for path in pathlib.Path(root_dir).glob(include): 
    paths.append(path)

  paths = sorted(paths)

  return paths

    
def base_filename_organization(imagPaths):
    baseFileNames = {}
    for imagPath in imagPaths:
        baseFileName = '_'.join(str(imagPath.stem).split('_')[:-1])
        if baseFileName not in baseFileNames:
            baseFileNames[baseFileName] = []
        baseFileNames[baseFileName].append(imagPath)

    # Sort lists numerically by the last suffix
    for key in baseFileNames:
        baseFileNames[key].sort(
            key=lambda p: int(p.stem.split('_')[-1])
        )

    return baseFileNames


def remove_last_element(baseFileNames):
    for baseFileName in baseFileNames:
        imagPaths = baseFileNames[baseFileName]
        length = len(imagPaths)
        baseFileNameLength = baseFileName + f'_{length-1}'
        imagPaths = [imagPath for imagPath in imagPaths if baseFileNameLength not in str(imagPath)]
        baseFileNames[baseFileName] = imagPaths
    return baseFileNames


def make_composite_images(images_dict: dict, final_width: int, final_height: int, save_dir: str):
    """
    Creates composite images from dict of images using OpenCV.

    Args:
        images_dict (dict): {key: [list of image paths]}
        final_width (int): width of final composite image
        final_height (int): height of final composite image
        save_dir (str): directory to save composite results
    """
    pathlib.Path(save_dir).mkdir(parents=True, exist_ok=True)

    for key, img_paths in images_dict.items():
        n = len(img_paths)
        grid_size = math.ceil(math.sqrt(n))  # square-ish grid
        cell_w = final_width // grid_size
        cell_h = final_height // grid_size

        # Black canvas
        composite = np.zeros((final_height, final_width, 3), dtype=np.uint8)

        for idx, img_path in enumerate(img_paths):
            try:
                img = cv2.imread(str(img_path))
                if img is None:
                    print(f"⚠️ Could not load {img_path}")
                    continue

                img = cv2.resize(img, (cell_w, cell_h), interpolation=cv2.INTER_AREA)

                row, col = divmod(idx, grid_size)
                y, x = row * cell_h, col * cell_w
                composite[y:y+cell_h, x:x+cell_w] = img
            except Exception as e:
                print(f"⚠️ Error with {img_path}: {e}")

        out_path = str(pathlib.Path(save_dir) / f"{key}.jpg")
        cv2.imwrite(out_path, composite)
        print(f"✅ Saved {out_path}")


In [7]:
imagsPaths = {}
imagsPaths['fall']    = get_image_paths( ROOT_DIR_IMAGES + '/Fall/Raw_Image')
imagsPaths['no_fall'] = get_image_paths( ROOT_DIR_IMAGES + '/No_Fall/Raw_Image')

print(f"[INFO] videos dict keys: {list(imagsPaths.keys())}")
print(f"[INFO] videos['fall'] has {len(imagsPaths['fall'])} entries")
print(f"[INFO] videos['no_fall'] has {len(imagsPaths['no_fall'])} entries")

[INFO] videos dict keys: ['fall', 'no_fall']
[INFO] videos['fall'] has 49985 entries
[INFO] videos['no_fall'] has 61029 entries


In [8]:
BASEFILENAMES = {}

BASEFILENAMES['fall'] = base_filename_organization(imagsPaths['fall'])
BASEFILENAMES['no_fall'] = base_filename_organization(imagsPaths['no_fall'])
print( f"Fall {len(BASEFILENAMES['fall'])}")
print( f"No fall {len(BASEFILENAMES['no_fall'])}")


Fall 3136
No fall 3822


## Remove last image

In [9]:
BASEFILENAMES['fall'] = remove_last_element(BASEFILENAMES['fall'])
BASEFILENAMES['no_fall'] = remove_last_element(BASEFILENAMES['no_fall'])
print( f"Fall {len(BASEFILENAMES['fall'])}")
print( f"No fall {len(BASEFILENAMES['no_fall'])}")


Fall 3136
No fall 3822


In [10]:
make_composite_images(BASEFILENAMES['fall'],  final_width=800, final_height=800, save_dir='../Kaggle/Fall/Square_Image')

✅ Saved ../Kaggle/Fall/Square_Image/20240912_101331.jpg
✅ Saved ../Kaggle/Fall/Square_Image/20240912_101427.jpg
✅ Saved ../Kaggle/Fall/Square_Image/20240912_101520.jpg
✅ Saved ../Kaggle/Fall/Square_Image/20240912_101626.jpg
✅ Saved ../Kaggle/Fall/Square_Image/20240912_101723.jpg
✅ Saved ../Kaggle/Fall/Square_Image/20240912_101943.jpg
✅ Saved ../Kaggle/Fall/Square_Image/20240912_102048.jpg
✅ Saved ../Kaggle/Fall/Square_Image/20240912_102146.jpg
✅ Saved ../Kaggle/Fall/Square_Image/20240912_102330.jpg
✅ Saved ../Kaggle/Fall/Square_Image/20240912_102649.jpg
✅ Saved ../Kaggle/Fall/Square_Image/20240912_102739.jpg
✅ Saved ../Kaggle/Fall/Square_Image/20240912_102926.jpg
✅ Saved ../Kaggle/Fall/Square_Image/20240912_103041.jpg
✅ Saved ../Kaggle/Fall/Square_Image/20240912_103200.jpg
✅ Saved ../Kaggle/Fall/Square_Image/20240912_103345.jpg
✅ Saved ../Kaggle/Fall/Square_Image/20240912_103443.jpg
✅ Saved ../Kaggle/Fall/Square_Image/20240912_103526.jpg
✅ Saved ../Kaggle/Fall/Square_Image/20240912_103

In [11]:
make_composite_images(BASEFILENAMES['no_fall'],  final_width=800, final_height=800, save_dir='../Kaggle/No_Fall/Square_Image')

✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0001.jpg
✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0002.jpg
✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0003.jpg
✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0004.jpg
✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0005.jpg
✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0006_resized.jpg
✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0007_resized.jpg
✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0008_resized.jpg
✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0009_resized.jpg
✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0010_resized.jpg
✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0011.jpg
✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0012.jpg
✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0013.jpg
✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0014.jpg
✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0015.jpg
✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0016.jpg
✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0017.jpg
✅ Saved ../Kaggle/No_Fall/Square_Image/B_D_0018.jpg
✅ Saved ../Kaggle/No_Fal